In [10]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [11]:
from langchain_core.documents import Document

In [12]:
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)

In [13]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [14]:
type(sample_doc)

langchain_core.documents.base.Document

In [15]:
# Text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/Python.txt", encoding="utf-8")

/tmp/ipykernel_4783/2888318756.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [16]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'Python (2).txt', 'research2 (1).pdf', 'Python (1).txt', 'data', 'Python.txt', 'sample_data']


In [17]:
import os

os.makedirs("/content/data", exist_ok=True)

with open("/content/data/Python.txt", "w", encoding="utf-8") as f:
    f.write("Python is a programming language. It is used in AI and machine learning.")

print("File created successfully!")

File created successfully!


In [18]:
from google.colab import files

uploaded = files.upload()

Saving research2.pdf to research2.pdf


In [19]:
from google.colab import files

uploaded = files.upload()

Saving Python.txt to Python (3).txt


In [20]:
import os

print("Files in /content:", os.listdir("/content"))

Files in /content: ['.config', 'Python (2).txt', 'research2 (1).pdf', 'research2.pdf', 'Python (3).txt', 'Python (1).txt', 'data', 'Python.txt', 'sample_data']


In [21]:
import os
import shutil

pdf_folder = "/content/data/pdfs"
os.makedirs(pdf_folder, exist_ok=True)

shutil.move(
    "/content/research2.pdf",
    "/content/data/pdfs/research2.pdf"
)

print("PDF folder contents:", os.listdir(pdf_folder))

PDF folder contents: ['research2.pdf']


In [22]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("/content/data/Python.txt")
documents = loader.load()

print(documents)

[Document(metadata={'source': '/content/data/Python.txt'}, page_content='Python is a programming language. It is used in AI and machine learning.')]


In [23]:

import os
from langchain_community.document_loaders.pdf import PyPDFLoader

def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))

    return all_docs


In [24]:
all_pdf_documents = load_all_pdfs()

total pdfs: 1
total pages: 21


In [25]:
print(all_pdf_documents[0].page_content[:1000])

1
Retrieval-Augmented Generation for Large
Language Models: A Survey
Yunfan Gaoa, Yun Xiong b, Xinyu Gao b, Kangxiang Jia b, Jinliu Pan b, Yuxi Bi c, Yi Dai a, Jiawei Sun a, Meng
Wangc, and Haofen Wang a,c
aShanghai Research Institute for Intelligent Autonomous Systems, Tongji University
bShanghai Key Laboratory of Data Science, School of Computer Science, Fudan University
cCollege of Design and Innovation, Tongji University
Abstract—Large Language Models (LLMs) showcase impres-
sive capabilities but encounter challenges like hallucination,
outdated knowledge, and non-transparent, untraceable reasoning
processes. Retrieval-Augmented Generation (RAG) has emerged
as a promising solution by incorporating knowledge from external
databases. This enhances the accuracy and credibility of the
generation, particularly for knowledge-intensive tasks, and allows
for continuous knowledge updates and integration of domain-
specific information. RAG synergistically merges LLMs’ intrin-
sic knowledge 

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [27]:
chunks = split_docs(all_pdf_documents)

In [28]:
len(chunks)

244

In [29]:
from sentence_transformers import SentenceTransformer

In [30]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [31]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


/tmp/ipykernel_4783/1519971626.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


In [32]:
import chromadb
import uuid

In [33]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [34]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 244


In [35]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

embeddings shape: (244, 384)
total documents added in vector store= 244
docs in collection: 488


In [36]:
from sklearn.metrics.pairwise import cosine_similarity

In [37]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [38]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [39]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 0 documents


[]

In [40]:
print("PDF pages:", len(all_pdf_documents))
print("Chunks:", len(chunks))

if chunks:
    print("First chunk:")
    print(chunks[0].page_content[:300])

PDF pages: 21
Chunks: 244
First chunk:
1
Retrieval-Augmented Generation for Large
Language Models: A Survey
Yunfan Gaoa, Yun Xiong b, Xinyu Gao b, Kangxiang Jia b, Jinliu Pan b, Yuxi Bi c, Yi Dai a, Jiawei Sun a, Meng
Wangc, and Haofen Wang a,c
aShanghai Research Institute for Intelligent Autonomous Systems, Tongji University
bShanghai K


In [42]:
%whos

Variable                         Type                  Data/Info
----------------------------------------------------------------
API_KEY_OPENAI                   str                   sk-proj-QLSgN26C9N4CTsfzA<...>oSf-DYYEoVU4WM4Ygv87S4zEA
ChatOpenAI                       ModelMetaclass        <class 'langchain_openai.<...>_models.base.ChatOpenAI'>
Document                         ModelMetaclass        <class 'langchain_core.documents.base.Document'>
EmbeddingManager                 type                  <class '__main__.EmbeddingManager'>
PyPDFLoader                      ABCMeta               <class 'langchain_communi<...>loaders.pdf.PyPDFLoader'>
RAGRetriever                     type                  <class '__main__.RAGRetriever'>
RecursiveCharacterTextSplitter   ABCMeta               <class 'langchain_text_sp<...>veCharacterTextSplitter'>
SentenceTransformer              ABCMeta               <class 'sentence_transfor<...>del.SentenceTransformer'>
TextLoader                       

In [43]:
print(type(vector_store))
print(dir(vector_store))

<class '__main__.VectorStoreManager'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_initialize_store', 'add_documents', 'client', 'collection', 'collection_name', 'persist_directory']


In [44]:
print("Collection name:", vector_store.collection_name)
print("Stored documents:", vector_store.collection.count())

Collection name: pdf_documents
Stored documents: 488


In [45]:
data = vector_store.collection.get(
    limit=2,
    include=["documents"]
)

print("Stored text:")
for doc in data["documents"]:
    print(doc[:300])
    print("-" * 40)

Stored text:
1
Retrieval-Augmented Generation for Large
Language Models: A Survey
Yunfan Gaoa, Yun Xiong b, Xinyu Gao b, Kangxiang Jia b, Jinliu Pan b, Yuxi Bi c, Yi Dai a, Jiawei Sun a, Meng
Wangc, and Haofen Wang a,c
aShanghai Research Institute for Intelligent Autonomous Systems, Tongji University
bShanghai K
----------------------------------------
sive capabilities but encounter challenges like hallucination,
outdated knowledge, and non-transparent, untraceable reasoning
processes. Retrieval-Augmented Generation (RAG) has emerged
as a promising solution by incorporating knowledge from external
databases. This enhances the accuracy and credibi
----------------------------------------


In [47]:
print("Retriever type:", type(rag_retriever))
print("Retriever methods:")

print([
    name for name in dir(rag_retriever)
    if not name.startswith("__")
])

Retriever type: <class '__main__.RAGRetriever'>
Retriever methods:
['embedding_manager', 'retrieve', 'vector_store']


In [48]:
results = vector_store.collection.query(
    query_texts=["What is Retrieval-Augmented Generation?"],
    n_results=3
)

print("Retrieved documents:", len(results["documents"][0]))

for i, doc in enumerate(results["documents"][0], start=1):
    print(f"\nDocument {i}:")
    print(doc[:300])

Retrieved documents: 3

Document 1:
arXiv:2212.14024, 2022.
[24] Z. Jiang, F. F. Xu, L. Gao, Z. Sun, Q. Liu, J. Dwivedi-Yu, Y . Yang,
J. Callan, and G. Neubig, “Active retrieval augmented generation,”
arXiv preprint arXiv:2305.06983 , 2023.
[25] A. Asai, Z. Wu, Y . Wang, A. Sil, and H. Hajishirzi, “Self-rag:
Learning to retrieve, gene

Document 2:
arXiv:2212.14024, 2022.
[24] Z. Jiang, F. F. Xu, L. Gao, Z. Sun, Q. Liu, J. Dwivedi-Yu, Y . Yang,
J. Callan, and G. Neubig, “Active retrieval augmented generation,”
arXiv preprint arXiv:2305.06983 , 2023.
[25] A. Asai, Z. Wu, Y . Wang, A. Sil, and H. Hajishirzi, “Self-rag:
Learning to retrieve, gene

Document 3:
three sections explore core components—“Retrieval”, “Gen-
eration” and “Augmentation”, respectively. Section III focuses
on optimization methods in retrieval,including indexing, query
and embedding optimization. Section IV concentrates on post-
retrieval process and LLM fine-tuning in generation. Se


In [49]:
import inspect

print(inspect.signature(rag_retriever.retrieve))

(query, top_k=5, score_threshold=0.0)


In [50]:
result = rag_retriever.retrieve(
    "What is Retrieval-Augmented Generation?"
)

print("Result:", result)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents
Result: [{'id': 'doc_ed45a0f4-61c4-46c9-8677-881c800c1ae6', 'document': 'arXiv:2212.14024, 2022.\n[24] Z. Jiang, F. F. Xu, L. Gao, Z. Sun, Q. Liu, J. Dwivedi-Yu, Y . Yang,\nJ. Callan, and G. Neubig, “Active retrieval augmented generation,”\narXiv preprint arXiv:2305.06983 , 2023.\n[25] A. Asai, Z. Wu, Y . Wang, A. Sil, and H. Hajishirzi, “Self-rag:\nLearning to retrieve, generate, and critique through self-reflection,”\narXiv preprint arXiv:2310.11511 , 2023.\n[26] Z. Ke, W. Kong, C. Li, M. Zhang, Q. Mei, and M. Bendersky,', 'metadata': {'subject': '', 'doc_index': 175, 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'page_label': '17', 'author': '', 'moddate': '2024-03-28T00:54:45+00:00', 'creator': 'LaTeX with hyperref', 'total_pages': 21, 'producer': 'pdfTeX-1.40.25', 'source': 'data/pdfs/research2.pdf', 'title': '', 'content_length': 448, 'creationdate': '2024-03-28T00:54:4

In [51]:
print("Notebook objects related to generation:")

for name, obj in globals().copy().items():
    if any(word in name.lower() for word in [
        "llm", "generator", "generation", "chain", "qa", "answer"
    ]):
        print(name, type(obj))

Notebook objects related to generation:
llm <class 'langchain_openai.chat_models.base.ChatOpenAI'>


In [52]:
print("All notebook variables and objects:")

%whos

All notebook variables and objects:
Variable                         Type                  Data/Info
----------------------------------------------------------------
API_KEY_OPENAI                   str                   sk-proj-QLSgN26C9N4CTsfzA<...>oSf-DYYEoVU4WM4Ygv87S4zEA
ChatOpenAI                       ModelMetaclass        <class 'langchain_openai.<...>_models.base.ChatOpenAI'>
Document                         ModelMetaclass        <class 'langchain_core.documents.base.Document'>
EmbeddingManager                 type                  <class '__main__.EmbeddingManager'>
PyPDFLoader                      ABCMeta               <class 'langchain_communi<...>loaders.pdf.PyPDFLoader'>
RAGRetriever                     type                  <class '__main__.RAGRetriever'>
RecursiveCharacterTextSplitter   ABCMeta               <class 'langchain_text_sp<...>veCharacterTextSplitter'>
SentenceTransformer              ABCMeta               <class 'sentence_transfor<...>del.SentenceTransformer

In [54]:
print("EmbeddingManager methods:")

print([
    name for name in dir(embedding_manager)
    if not name.startswith("__")
])

EmbeddingManager methods:
['generate_embeddings', 'model', 'model_name']


In [55]:
import inspect

print(inspect.signature(embedding_manager.generate_embeddings))

(text)


In [56]:
print("Model name:", embedding_manager.model_name)
print("Model type:", type(embedding_manager.model))

Model name: all-MiniLM-L6-v2
Model type: <class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [57]:
test_embedding = embedding_manager.generate_embeddings(
    "What is Retrieval-Augmented Generation?"
)

print("Output type:", type(test_embedding))
print("Output shape:", test_embedding.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (384,)
Output type: <class 'numpy.ndarray'>
Output shape: (384,)


In [ ]:
##INTEGRATE WITH LLM

In [ ]:
##groq

In [4]:
!pip install langchain-groq

In [5]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


In [16]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

In [17]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [18]:
from langchain_core.messages import HumanMessage

response = llm.invoke([HumanMessage(content="What is RAG?")])
print(response.content)

**RAG** most commonly stands for **Retrieval‑Augmented Generation**—a technique used in natural‑language processing (NLP) to combine large language models (LLMs) with external knowledge sources.  

---

## 1. What is Retrieval‑Augmented Generation?

| Component | What it does | Why it matters |
|-----------|--------------|----------------|
| **Retriever** | Searches a large corpus (e.g., Wikipedia, company docs, PDFs) for passages relevant to the user’s query. | Provides up‑to‑date, domain‑specific facts that the LLM might not have memorized. |
| **Generator** | A language model (e.g., GPT‑4, LLaMA) that takes the retrieved passages as context and produces a fluent answer. | Leverages the LLM’s language skills while grounding the response in real data. |
| **Fusion** | The process of feeding retrieved text into the generator, often via prompt engineering or a neural fusion layer. | Ensures the model can “see” the evidence before answering. |

### How it works (high‑level flow)

1. **Us

In [10]:
response = llm.invoke("What is RAG?")
print(response.content)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}